# GIK-IceChain - Colab run: C1 (ingestion) + C2 (exceedance)

Runs **the first two pipeline stages on Colab**, writing to the MinIO store:

1. **C1 - `convert`**: IFS ENS GRIB2 ingestion into the source IceChunk store.
2. **C2 - `exceedance`**: accumulations + adaptive GEV exceedance into `exceedance-zarr` on MinIO.

**C3 (risk) runs locally** - see `scripts/run_c3_local.sh` (reads `exceedance-zarr` from MinIO).

## Target window: the full available corpus

The target window is **the entire GIK catalog coverage** (the source of C1's byte-range
references): **2024-03-01 to 2026-02-18, 720 days, no gaps**. The "Coverage and block plan"
cell recomputes it dynamically each session. Source GRIBs on `s3://ecmwf-forecasts` are
verified available **beyond the assumed 15-month retention** (anonymous HTTP 200 observed
back to February 2023).

Note: the **0p4 era** (2023-01-18 to 2024-02-28, 0.4-degree grid) is not in the GIK
catalog and therefore not ingestible by C1 as-is. It is readable through the mentor's
**Demo 6 virtual store** (source.coop `e4drr-project/forecasts/`, anonymous access,
`gribberish` codec) - see `docs/ISSUES.md`.

### MinIO credentials
Read in order: **Colab Secrets** (`MINIO`, `MINIO_ACCESS_KEY`, `MINIO_SECRET_KEY`), env vars, then `.env`. On Colab, add them in the *Secrets* tab. No synthetic fallback: if the store is unreachable, cells raise.


## 0. Parameters - edit here

**Colab session limit.** The full corpus is ~720 days; C1 (GRIB re-download ~1 GB/day) plus C2 far exceed one Colab session (12 h free / 24 h Pro). **Run in sub-blocks**: leave `START = "auto"` (the notebook picks the next block of `BLOCK_DAYS` missing days from the store), or set `START`/`END` manually. C1 is idempotent (`append`) and C2 skips dates already written, so the run resumes cleanly block by block, session after session.


In [ ]:
# Global target: the whole GIK catalog (2024-03-01 to 2026-02-18, ~720 days).
# START = "auto": the coverage cell picks the next block of BLOCK_DAYS
# not-yet-committed days from the C1 store.
START      = "auto"                # "auto" or "YYYY-MM-DD" (block start, incl.)
END        = ""                    # "" = START + BLOCK_DAYS - 1, or "YYYY-MM-DD" (incl.)
BLOCK_DAYS = 7
RUN_C1     = True
RUN_C2     = True
C2_WORKERS = 1                     # 1 = sequential (~6 GB RAM, robust); raise on bigger VMs
CONFIG     = "configs/default.yaml"
print(f"Block START={START!r} END={END!r} BLOCK_DAYS={BLOCK_DAYS} | "
      f"RUN_C1={RUN_C1} RUN_C2={RUN_C2} workers={C2_WORKERS}")

## 1. Setup - repo, install, MinIO creds, prerequisites

In [ ]:
import os, sys, subprocess
from pathlib import Path


def _bootstrap() -> Path:
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "configs" / "default.yaml").exists():
            return p
    repo = Path.cwd() / "gik-icechain"
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/hashirama21/gik-icechain.git", str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{repo}[dev]"], check=True)
    return repo


REPO = _bootstrap()
DATA = REPO / "data"
sys.path.insert(0, str(REPO / "src"))


def _secret(name: str, default: str = "") -> str:
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    return os.environ.get(name, default)


_env = REPO / ".env"
if _env.exists():
    for _l in _env.read_text().splitlines():
        if _l and not _l.startswith("#") and "=" in _l:
            _k, _v = _l.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip())

_minio = _secret("MINIO") or _secret("MINIO_ENDPOINT_URL")
ENDPOINT = _minio if _minio.startswith("http") else (f"http://{_minio}" if _minio else "")
KEY = _secret("MINIO_ACCESS_KEY") or _secret("AWS_ACCESS_KEY_ID")
SECRET = _secret("MINIO_SECRET_KEY") or _secret("AWS_SECRET_ACCESS_KEY")
if not (ENDPOINT and KEY and SECRET):
    raise RuntimeError(
        "MinIO creds required (MINIO/MINIO_ACCESS_KEY/MINIO_SECRET_KEY via Colab Secrets, env, or .env).")

# CLI calls run as subprocesses and re-read the config (empty endpoint):
# the MinIO endpoint is resolved through these env vars (inherited via env=os.environ).
os.environ["AWS_ENDPOINT_URL"] = ENDPOINT
os.environ["AWS_ACCESS_KEY_ID"] = KEY
os.environ["AWS_SECRET_ACCESS_KEY"] = SECRET
os.environ.setdefault("AWS_REGION", "eu-west-1")
os.environ.setdefault("ECCODES_PYTHON_USE_FINDLIBS", "1")

if not (DATA / "cmorph_thresholds").exists():
    tools = [sys.executable, str(REPO / "scripts" / "tools.py")]
    subprocess.run([*tools, "download", "--component", "all"], check=True)
    subprocess.run([*tools, "download-thresholds"], check=True)

from gik_icechain.shared.config import load_config

cfg = load_config(REPO / CONFIG)
cfg.outputs.endpoint_url = ENDPOINT
STORAGE_OPTIONS = {"endpoint_url": ENDPOINT}
print("Repo            :", REPO)
print("MinIO           :", ENDPOINT)
print("Source store    :", cfg.outputs.icechunk_store_uri)
print("Exceedance store:", cfg.outputs.exceedance_store_uri)

## 1bis. Coverage and block plan

Compares **the GIK catalog** (target window = the full available corpus) with the **C1
store** (dates already committed): logs coverage, the list of missing days, and, when
`START="auto"`, resolves `[START, END]` to the **next contiguous missing block**.
Just rerun the notebook each session: it advances on its own.

In [ ]:
import datetime as dt

from gik_icechain.conversion.gik_loader import GIKCatalog
from gik_icechain.conversion.icechunk_writer import IceChainStore

catalog_dates = GIKCatalog().list_available_dates()
FULL_START, FULL_END = catalog_dates[0], catalog_dates[-1]
print(f"GIK catalog : {len(catalog_dates)} days  {FULL_START} -> {FULL_END}")

_store = IceChainStore(cfg.outputs.icechunk_store_uri,
                       region=cfg.outputs.icechunk_store_region,
                       endpoint_url=ENDPOINT)
_store.create_or_open()
committed = {s["forecast_date"] for s in _store.list_snapshots() if s["forecast_date"]}
in_target = {d for d in committed if FULL_START.isoformat() <= d <= FULL_END.isoformat()}
missing = [d for d in catalog_dates if d.isoformat() not in committed]
pct = 100.0 * len(in_target) / len(catalog_dates)
print(f"C1 store    : {len(in_target)}/{len(catalog_dates)} days committed ({pct:.1f} %)")
if missing:
    print(f"Missing     : {len(missing)} days  (first {missing[0]}, last {missing[-1]})")

if str(START).lower() == "auto":
    if not missing:
        raise SystemExit("Corpus complete: nothing to ingest.")
    _s = _e = missing[0]
    for d in missing[1: BLOCK_DAYS]:
        if d != _e + dt.timedelta(days=1):
            break
        _e = d
    START, END = _s.isoformat(), _e.isoformat()
elif not END:
    _s = dt.date.fromisoformat(START)
    END = min(_s + dt.timedelta(days=BLOCK_DAYS - 1), FULL_END).isoformat()

n_block = (dt.date.fromisoformat(END) - dt.date.fromisoformat(START)).days + 1
new_days = sum(1 for i in range(n_block)
               if (dt.date.fromisoformat(START) + dt.timedelta(days=i)).isoformat()
               not in committed)
print(f"Block       : {START} -> {END}  ({n_block} days, {n_block - new_days} already committed)")
print(f"After block : ~{pct + 100.0 * new_days / len(catalog_dates):.1f} % of corpus")

## 2. C1 - ingestion (`convert`) of the block

Ingests the whole `[START, END]` block into the source IceChunk store. `convert` is
idempotent (`append` via `create_or_open`). Per-day progress is logged by the CLI;
the cell adds wall-clock timing and a commit delta at the end.

In [ ]:
import time

if RUN_C1:
    cmd = [sys.executable, "-m", "gik_icechain", "convert",
           "--start", START, "--end", END, "--config", str(REPO / CONFIG)]
    print("C1:", " ".join(cmd), flush=True)
    t0 = time.time()
    rc = subprocess.run(cmd, env=os.environ).returncode
    elapsed = time.time() - t0
    assert rc == 0, f"C1 convert failed (exit {rc})"

    _store.open()
    now_committed = {s["forecast_date"] for s in _store.list_snapshots() if s["forecast_date"]}
    added = sorted(now_committed - committed)
    committed = now_committed
    print(f"C1 OK in {elapsed/60:.1f} min "
          f"({elapsed/max(len(added), 1)/60:.1f} min/day, {len(added)} new days)")
    print("Committed this block:", ", ".join(added) if added else "none (already present)")
else:
    print("C1 skipped (RUN_C1=False).")

## 3. C2 - exceedance

Reads the source IceChunk store, computes adaptive GEV exceedance for `[START, END]`,
writes `exceedance-zarr` on MinIO. Dates already present are skipped on append.

In [ ]:
if RUN_C2:
    cmd = [sys.executable, "-m", "gik_icechain", "exceedance",
           "--store", cfg.outputs.icechunk_store_uri,
           "--output", cfg.outputs.exceedance_store_uri,
           "--start", START, "--end", END,
           "--workers", str(C2_WORKERS),
           "--config", str(REPO / CONFIG)]
    print("C2:", " ".join(cmd), flush=True)
    t0 = time.time()
    rc = subprocess.run(cmd, env=os.environ).returncode
    elapsed = time.time() - t0
    assert rc == 0, f"C2 exceedance failed (exit {rc})"
    print(f"C2 OK in {elapsed/60:.1f} min")
else:
    print("C2 skipped (RUN_C2=False).")

## 4. Verification - exceedance store and anti-corruption guard

Checks the written dates and that **all variables share the same `date` dimension**
(the bug that once froze `median_ratio`/`tail_ratio` at 4 dates while
`exceedance_prob` grew to 24; the writer now pads on append).

In [ ]:
import numpy as np
import xarray as xr

exc = xr.open_zarr(cfg.outputs.exceedance_store_uri, consolidated=False,
                   storage_options=STORAGE_OPTIONS)
dates = [str(d)[:10] for d in np.sort(exc["date"].values)]
print(f"Exceedance store: {len(dates)} dates  {dates[0]}..{dates[-1]}")
print("Variables       :", list(exc.data_vars))
nd = exc.sizes["date"]
desync = {v: int(exc[v].sizes.get("date", 0)) for v in exc.data_vars
          if exc[v].sizes.get("date") != nd}
if desync:
    print("Date DESYNC detected:", desync, "-> corrupted store (see writer guard).")
else:
    print(f"OK - all variables aligned on date={nd}.")

## 4bis. Storage footprint - per day and total

Measures the bytes actually stored on MinIO for both stores (`s3 du` over each prefix):

- **C1 IceChunk store**: metadata only (snapshots + manifests of virtual byte-range
  references); the GRIB bytes stay on `s3://ecmwf-forecasts`. The cell also reports the
  virtual-vs-stored amplification ratio.
- **C2 exceedance-zarr**: real computed chunks, broken down per variable.

Per-day figures are averages (total / committed days): IceChunk deduplicates across
snapshots and zarr chunks span 30 dates, so exact per-day attribution is not defined.

In [ ]:
import s3fs


def _fmt(n: float) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or unit == "TB":
            return f"{n:,.1f} {unit}"
        n /= 1024
    return f"{n:,.1f} TB"


fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": ENDPOINT})


def _du(uri: str) -> tuple[int, int]:
    sizes = fs.du(uri.removeprefix("s3://"), total=False)
    return sum(sizes.values()), len(sizes)


c1_uri = cfg.outputs.icechunk_store_uri
c1_bytes, c1_objects = _du(c1_uri)
n_days = len(committed)
print(f"C1 IceChunk store  {c1_uri}")
print(f"  total    : {_fmt(c1_bytes)}  ({c1_objects:,} objects, {n_days} days)")
print(f"  per day  : {_fmt(c1_bytes / max(n_days, 1))} (average)")

# Virtual footprint: bytes referenced on ecmwf-forecasts vs metadata stored.
GRIB_BYTES_PER_DAY = 1.2e9  # 51 members x 85 steps x 5 vars, EA byte ranges
virtual = n_days * GRIB_BYTES_PER_DAY
if c1_bytes:
    print(f"  virtual  : ~{_fmt(virtual)} referenced "
          f"(amplification ~{virtual / c1_bytes:,.0f}x)")

exc_uri = cfg.outputs.exceedance_store_uri
exc_bytes, exc_objects = _du(exc_uri)
exc_days = exc.sizes["date"]
print(f"\nC2 exceedance-zarr {exc_uri}")
print(f"  total    : {_fmt(exc_bytes)}  ({exc_objects:,} objects, {exc_days} days)")
print(f"  per day  : {_fmt(exc_bytes / max(exc_days, 1))} (average)")
for var in exc.data_vars:
    v_bytes, _ = _du(f"{exc_uri}/{var}")
    print(f"    {var:22s} {_fmt(v_bytes)}")

print(f"\nMinIO grand total  : {_fmt(c1_bytes + exc_bytes)}")

## 5. C3 - locally

C3 (CRMA risk) does not run here: launch it on your machine once C2 has written the
window you need. It reads `exceedance-zarr` from MinIO.

```bash
# from the repo, locally (git bash):
bash scripts/run_c3_local.sh 2024-03-01 2026-02-18
```

The script reads MinIO creds from `.env`, runs `gik-icechain risk` over the window,
and writes GeoJSON files to `results/admin1_risk/`.
